> 在线讲义：[第 63 章 · 共享资源池与 RL 资源配比：资源够不够，还要看在不在一起](https://wikieden.github.io/ai-infra-from-constraints/lessons/63-resource-pool.html) · 本书《从约束推导 AI Infra》

# 第 63 章 · 共享资源池与 RL 资源配比：资源够不够，还要看在不在一起

配套讲义：仓库里 `ai-infra/notes/lessons/63-resource-pool.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



资源碎片与迁移、突发排队、RL 迭代各段、权重分发的共享出口、临时实例窗口与成本、验证批次的长尾与最迟开始，取自 AII 第 11 章 §11.3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `place_options`**：小工具。

In [ ]:
def place_options(wait: float = 12, move: float = 4, run_local: float = 20, ready_spread: float = 1, run_spread: float = 32) -> dict[str, float]:
    return {"等待": wait + run_local, "迁移": move + run_local, "分散": ready_spread + run_spread}

**函数 `burst_waits`**：n 个同时到达、每个 service 秒的工作，cores 个执行者按 FIFO。

In [ ]:
def burst_waits(n: int, cores: int, service: float = 1.0) -> list[float]:
    """n 个同时到达、每个 service 秒的工作，cores 个执行者按 FIFO。"""
    return [(i // cores) * service for i in range(n)]

**函数 `iteration`**：小工具。

In [ ]:
def iteration(gen: float = 40, verify: float = 10, update: float = 30, publish: float = 5, gen_speed: float = 1) -> float:
    return gen / gen_speed + verify + update + publish

**函数 `broadcast_s`**：T_all ≥ max(单份 / 接收带宽, n 份 / 发送出口)。

In [ ]:
def broadcast_s(bytes_each: float, n: int, recv_bw: float, send_bw: float) -> float:
    """T_all ≥ max(单份 / 接收带宽, n 份 / 发送出口)。"""
    return max(bytes_each / recv_bw, n * bytes_each / send_bw)

**函数 `window`**：小工具。

In [ ]:
def window(L: float, T0: float) -> float:
    return max(L - T0, 0.0)

**函数 `cost_ratio`**：单位工作成本相对原来的倍数。

In [ ]:
def cost_ratio(base: float, extra: float, speedup: float) -> float:
    """单位工作成本相对原来的倍数。"""
    return (base + extra) / base / speedup

**函数 `residual`**：E[S − e | S > e]。

In [ ]:
def residual(samples: list[float], e: float) -> float:
    """E[S − e | S > e]。"""
    alive = [s for s in samples if s > e]
    return sum(s - e for s in alive) / len(alive)

**函数 `batch_done`**：单个验证进程：逐条提交 vs 等全部就绪。

In [ ]:
def batch_done(arrivals: list[float], work: float, eager: bool = True) -> float:
    """单个验证进程：逐条提交 vs 等全部就绪。"""
    t = 0.0 if eager else max(arrivals)
    for a in arrivals:
        t = max(t, a) + work
    return t

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：碎片与迁移")
print(f"  {place_options()}；迁移要 12 s 时与等待打平")
print("  ASI 加可抢占低优先级作业：GPU 分配 68% → 93%")

In [ ]:
print("\n第二步：突发与优先级")
w = burst_waits(10, 1)
print(f"  十个同时到达、一个执行者：平均等 {sum(w) / 10:.1f} s")
w = burst_waits(80, 48)
print(f"  80 个工具调用、48 核：{sum(1 for x in w if x > 0)} 个等 1 s，平均 {sum(w) / 80:.1f} s；96 核全在 1 s 内；80 个环境 {80 * 2} GiB")
print("  在线最多等 2 s、离线保存 30 s → 必须另留能 2 s 内交出的资源；SCHED_IDLE 下 LS 慢 45%，加 core scheduling 17%")

In [ ]:
print("\n第三步：RL 迭代")
print(f"  40 + 10 + 30 + 5 = {iteration():.0f} s；生成快一倍 {iteration(gen_speed=2):.0f} s（{iteration() / iteration(gen_speed=2):.2f} 倍）；生成为零上限 {iteration() / iteration(gen_speed=1e9):.2f} 倍")
W = 16_381_470_720
print(f"  Qwen3-8B 权重 {W / 1e9:.2f} GB 发 6 份：单份 {W / 6.25e9:.2f} s，全部 ≥ {broadcast_s(W, 6, 6.25e9, 25e9):.2f} s")
print(f"  临时实例准备 8 s：可用 20 s → {window(20, 8):.0f} s 生成；可用 5 s → {window(5, 8):.0f}")
print(f"  训练 84 $/h + 6 × 5.3 $/h = {84 + 6 * 5.3:.1f}：吞吐要 {(84 + 31.8) / 84:.2f} 倍；1.2 倍 → 成本 {cost_ratio(84, 31.8, 1.2):.2f}，1.6 倍 → {cost_ratio(84, 31.8, 1.6):.2f}")
print("  PolyRL 截断：4000 与 1000 token → 各取 1000，复用 2000、重生成 3000")

In [ ]:
print("\n第四步：验证")
print(f"  第 0/10/20 s 到达、每条 10 s：逐条提交 {batch_done([0, 10, 20], 10):.0f} s，等齐再做 {batch_done([0, 10, 20], 10, False):.0f} s")
s = [1] * 9 + [100]
print(f"  九条 1 s、一条 100 s：均值 {sum(s) / 10} s；已跑 10 s 的剩余 {residual(s, 10):.0f} s（不是 0.9）")
print("  整批最早 120 s 完成：100 s 的活最迟第 20 s 开始，准备 5 s 则第 15 s")
for g in (64, 16):
    print(f"  验证省 60 H100·秒、训练组 {g} 卡多等 2 s：净 {g * 2 - 60:+d} H100·秒")

## 3. 代码里才看得到的

1. `burst_waits` 是 FIFO：第 i 个等 ⌊i / 核数⌋ 个服务时间；80 个在 48 核上正好 32 个等 1 s。
2. `broadcast_s` 只是下界，没计启动、加载进显存和确认版本。
3. `cost_ratio` 假设所有实例全程计费；临时实例只在用时计费时门槛会更低。

## 4. 自己改一改

- 给 `iteration` 加"验证与生成重叠"：一轮 = max(生成, 验证) + 更新 + 发布。
- 用 `residual` 试几何分布（无记忆），看剩余时间是否与已跑时间无关。
- 用 `broadcast_s` 改成树形分发（接收方收到后转发），估计 6 个实例的时间。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 空座位散在三张桌，四口之家坐不下；挪一个人还是等一桌散，比谁先凑齐；
> 十个人同时进门，平均等四分半；
> 只给一道工序加人，别的几道封顶；新人上岗先算交接时间；
> 等最慢那块肉，煮得久的还要煮更久。